# MAST IQ — Model Training Pipeline v3
### Per-Target XGBoost with Optuna Hyperparameter Optimization

**Dataset**: `steel_heat_treatment.csv` — 570 samples, 4 heat treatment processes

**Features (31 total)**:
- 10 elemental composition (C, Si, Mn, P, S, Ni, Cr, Cu, Mo, **Fe**)
- 7 engineered (Carbon_Equiv, A3_Temp, Delta_HT_A3, Hollomon_Jaffe, C×Cr, **Cooling_Rate_Est**, **Total_Alloy**)
- 4 process parameters (HT_Temp, Soak_Time, Tempering_Temp, Tempering_Time)
- 4 process dummies + 6 cooling medium dummies

**Targets**: Tensile (MPa) · Yield (MPa) · Hardness (HB) · Elongation (%) · Fatigue (MPa)

**Improvements over v2**:
- Fe (iron balance) added as a feature
- Cooling rate estimate and total alloy content as engineered features
- Per-target Optuna Bayesian optimization (80 trials × 5-fold CV per target)
- Early stopping to prevent overfitting
- Adjusted R² and comprehensive cross-validation metrics
- SHAP feature importance analysis

## 0  Install Dependencies (Colab)

In [ ]:
# Uncomment the line below if running on Google Colab
# !pip install xgboost optuna shap scikit-learn -q

## 1  Imports & Configuration

In [ ]:
import os, json, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from xgboost import XGBRegressor
import optuna
import shap

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams.update({
    "figure.facecolor": "#0e1117", "axes.facecolor": "#1a1d2e",
    "axes.edgecolor": "#334",      "axes.labelcolor": "#c8d8ff",
    "xtick.color": "#8a9bc0",      "ytick.color": "#8a9bc0",
    "text.color": "#c8d8ff",       "grid.color": "#2a3050",
    "grid.linestyle": "--",        "grid.alpha": 0.5,
    "font.family": "DejaVu Sans",  "figure.dpi": 110,
})
COLORS = ["#4c72b0", "#dd8452", "#55a868", "#c44e52", "#8172b3"]

# ── Config ──
DATA_FILE       = "steel_heat_treatment.csv"
OUTPUT_DIR      = "models"
TEST_SIZE       = 0.20
RANDOM_STATE    = 42
N_CV_FOLDS      = 5
OPTUNA_TRIALS   = 80    # per target (increase for better results, decrease to speed up)
EARLY_STOPPING  = 50    # XGBoost early stopping rounds

TARGET_COLS = ["Tensile_MPa", "Yield_MPa", "Hardness_HB", "Elongation_pct", "Fatigue_MPa"]

# Estimated cooling rates (°C/s) per medium — used as engineered feature
COOLING_RATES = {
    "Water": 80.0, "Oil": 25.0, "Polymer": 35.0,
    "Salt Bath": 15.0, "Air": 2.0, "Furnace": 0.1,
}

print("Imports OK")

## 2  Load Dataset

In [ ]:
df = pd.read_csv(DATA_FILE)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nProcess counts:")
print(df["Process"].value_counts())
print(f"\nCooling medium counts:")
print(df["Cooling_Medium"].value_counts())
df.head()

## 3  Feature Engineering

Physically meaningful derived features:
- **Fe** (iron balance) — the matrix element; its fraction directly affects lattice structure
- **Carbon Equivalent (IIW)** — hardenability proxy
- **A3 Temperature** — Andrews (1965) empirical austenite transformation
- **Delta HT-A3** — positive means fully austenitized
- **Hollomon-Jaffe** — tempering parameter
- **C × Cr** — carbide-forming interaction
- **Cooling Rate Estimate** — numerical proxy for cooling medium severity
- **Total Alloy Content** — sum of major alloying elements

In [ ]:
def carbon_equiv(row):
    """IIW carbon equivalent."""
    return row.C + row.Mn/6 + row.Si/24 + row.Ni/40 + row.Cr/5 + row.Mo/4 + row.Cu/15

def a3_temp(row):
    """Andrews (1965) A3 temperature."""
    return 912.0 - 203.0*np.sqrt(max(row.C, 1e-6)) - 30.0*row.Mn + 44.7*row.Si - 15.2*row.Ni + 31.5*row.Mo

def hollomon_jaffe(row):
    """Tempering parameter H = T(K) × (18 + log10(t_hours))."""
    if row.Tempering_Temp_C <= 0 or row.Tempering_Time_min <= 0:
        return 0.0
    t_h = max(row.Tempering_Time_min / 60.0, 0.001)
    return (row.Tempering_Temp_C + 273.15) * (18.0 + np.log10(t_h))

# Fe content (balance element)
alloy_cols = ["C", "Si", "Mn", "P", "S", "Ni", "Cr", "Cu", "Mo"]
if "Fe" not in df.columns:
    df["Fe"] = 100.0 - df[alloy_cols].sum(axis=1)

df["Carbon_Equiv"]   = df.apply(carbon_equiv, axis=1)
df["A3_Temp_C"]      = df.apply(a3_temp, axis=1)
df["Delta_HT_A3"]    = df["HT_Temp_C"] - df["A3_Temp_C"]
df["Hollomon_Jaffe"] = df.apply(hollomon_jaffe, axis=1)
df["C_x_Cr"]         = df["C"] * df["Cr"]
df["Cooling_Rate_Est"]= df["Cooling_Medium"].map(COOLING_RATES).fillna(2.0)
df["Total_Alloy"]    = df[["C","Si","Mn","Ni","Cr","Mo","Cu"]].sum(axis=1)

print("Engineered features:")
eng = ["Fe", "Carbon_Equiv", "A3_Temp_C", "Delta_HT_A3", "Hollomon_Jaffe", "C_x_Cr", "Cooling_Rate_Est", "Total_Alloy"]
df[eng].describe().round(3)

## 4  Exploratory Data Analysis

In [ ]:
# Target statistics
print("Target property statistics:")
df[TARGET_COLS].describe().round(1)

In [ ]:
# Property distributions by process type
process_colors = {"Quench_Temper":"#4c72b0", "Normalizing":"#55a868",
                  "Annealing":"#dd8452", "Stress_Relief":"#c44e52"}
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for ax, tgt in zip(axes, TARGET_COLS):
    for proc, grp in df.groupby("Process"):
        ax.hist(grp[tgt], bins=28, alpha=0.55, label=proc,
                color=process_colors.get(proc, "#888"), edgecolor="none")
    ax.set_xlabel(tgt); ax.set_ylabel("Count")
    ax.set_title(f"Distribution of {tgt}", fontsize=9)
    ax.legend(fontsize=7)
axes[-1].set_visible(False)
plt.suptitle("Mechanical Property Distributions by Process", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Box plots — property vs process
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
proc_list = list(process_colors.keys())
for ax, tgt, col in zip(axes, TARGET_COLS, COLORS):
    data = [df[df["Process"]==p][tgt].values for p in proc_list]
    bplot = ax.boxplot(data, patch_artist=True,
                       medianprops=dict(color="white", linewidth=2))
    for patch, pc in zip(bplot["boxes"], process_colors.values()):
        patch.set_facecolor(pc + "88")
    ax.set_xticks(range(1, len(proc_list)+1))
    ax.set_xticklabels(proc_list, rotation=20, ha="right", fontsize=7)
    ax.set_ylabel(tgt, fontsize=9); ax.set_title(tgt, fontsize=9)
plt.suptitle("Properties by Process Type", fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Correlation heatmap (chemistry + process params + Fe vs targets)
num_cols = ["C","Si","Mn","Ni","Cr","Mo","Fe","Carbon_Equiv","Total_Alloy",
            "HT_Temp_C","Soaking_Time_min","Tempering_Temp_C","Tempering_Time_min",
            "Cooling_Rate_Est"] + TARGET_COLS
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, ax=ax, cmap="coolwarm", center=0,
            annot=True, fmt=".2f", annot_kws={"size": 6},
            linewidths=0.3, linecolor="#1a1d2e",
            cbar_kws={"shrink": 0.7})
ax.set_title("Correlation Heatmap (incl. Fe & Cooling Rate)", fontsize=12, pad=10)
plt.tight_layout(); plt.show()

In [ ]:
# Fe content distribution — verify it makes metallurgical sense
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["Fe"], bins=40, color="#c9a56b", edgecolor="none", alpha=0.8)
axes[0].set_xlabel("Fe (wt%)"); axes[0].set_ylabel("Count")
axes[0].set_title(f"Fe Distribution — range [{df['Fe'].min():.1f}, {df['Fe'].max():.1f}]", fontsize=10)
axes[1].scatter(df["Total_Alloy"], df["Fe"], s=8, alpha=0.5, c="#6faed9")
axes[1].set_xlabel("Total Alloy Content (wt%)"); axes[1].set_ylabel("Fe (wt%)")
axes[1].set_title("Fe vs Total Alloy Content", fontsize=10)
plt.tight_layout(); plt.show()

## 5  One-Hot Encoding & Train/Test Split

In [ ]:
df_enc = pd.get_dummies(df, columns=["Process", "Cooling_Medium"], drop_first=False)

FEATURE_COLS = [
    # Chemistry (10 elements including Fe)
    "C", "Si", "Mn", "P", "S", "Ni", "Cr", "Cu", "Mo", "Fe",
    # Engineered (7)
    "Carbon_Equiv", "A3_Temp_C", "Delta_HT_A3", "Hollomon_Jaffe",
    "C_x_Cr", "Cooling_Rate_Est", "Total_Alloy",
    # Process parameters (4)
    "HT_Temp_C", "Soaking_Time_min", "Tempering_Temp_C", "Tempering_Time_min",
    # Process dummies (4)
    "Process_Quench_Temper", "Process_Normalizing",
    "Process_Annealing", "Process_Stress_Relief",
    # Cooling medium dummies (6)
    "Cooling_Medium_Water", "Cooling_Medium_Oil", "Cooling_Medium_Polymer",
    "Cooling_Medium_Air", "Cooling_Medium_Furnace", "Cooling_Medium_Salt Bath",
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_enc.columns]
print(f"Total features: {len(FEATURE_COLS)}")
print(", ".join(FEATURE_COLS))

X = df_enc[FEATURE_COLS].astype(float)
y = df_enc[TARGET_COLS].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print(f"\nTrain: {X_train.shape}   Test: {X_test.shape}")

## 6  Per-Target Optuna Hyperparameter Optimization

Each target gets its own XGBRegressor with individually optimized hyperparameters via Bayesian search (TPE sampler). This is superior to the v2 approach of shared params across a `MultiOutputRegressor`.

In [ ]:
def make_objective(X_tr, y_tr_col):
    """Optuna objective for a single target."""
    def objective(trial):
        params = {
            "n_estimators":     trial.suggest_int("n_estimators", 100, 1200),
            "max_depth":        trial.suggest_int("max_depth", 3, 10),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.30, log=True),
            "subsample":        trial.suggest_float("subsample", 0.50, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.40, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
            "gamma":            trial.suggest_float("gamma", 0.0, 10.0),
            "reg_alpha":        trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda":       trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        }
        model = XGBRegressor(
            **params, objective="reg:squarederror",
            tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
        )
        kf = KFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        scores = cross_val_score(model, X_tr, y_tr_col, cv=kf, scoring="r2", n_jobs=-1)
        return scores.mean()
    return objective

In [ ]:
models = {}
best_params = {}
eval_results = []
cv_results = []

for tgt in TARGET_COLS:
    print(f"\n{'='*65}")
    print(f"  Training: {tgt}")
    print(f"{'='*65}")

    y_tr = y_train[tgt]
    y_te = y_test[tgt]

    # ── Optuna search ──
    print(f"  Optuna: {OPTUNA_TRIALS} trials, {N_CV_FOLDS}-fold CV...")
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study.optimize(
        make_objective(X_train, y_tr),
        n_trials=OPTUNA_TRIALS,
        show_progress_bar=True,
    )
    bp = study.best_params
    best_params[tgt] = bp
    print(f"  Best CV R²: {study.best_value:.4f}")
    print(f"  Best params: {json.dumps({k: round(v,6) if isinstance(v,float) else v for k,v in bp.items()}, indent=4)}")

    # ── Train final model with early stopping ──
    final_model = XGBRegressor(
        **bp, objective="reg:squarederror", tree_method="hist",
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
        early_stopping_rounds=EARLY_STOPPING,
    )
    final_model.fit(X_train, y_tr, eval_set=[(X_test, y_te)], verbose=False)
    models[tgt] = final_model

    # ── Evaluation ──
    y_pred = final_model.predict(X_test)
    r2   = r2_score(y_te, y_pred)
    mae  = mean_absolute_error(y_te, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_te, y_pred)))
    mape = mean_absolute_percentage_error(y_te, y_pred) * 100
    n, p = len(y_te), X_test.shape[1]
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    eval_results.append({
        "Target": tgt, "R²": round(r2, 4), "Adj_R²": round(adj_r2, 4),
        "MAE": round(mae, 2), "RMSE": round(rmse, 2), "MAPE (%)": round(mape, 2),
        "Best_Iter": getattr(final_model, 'best_iteration', bp.get('n_estimators')),
    })
    print(f"  Test R²={r2:.4f}  Adj_R²={adj_r2:.4f}  MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.2f}%")

    # ── Cross-validation ──
    cv_model = XGBRegressor(
        **bp, objective="reg:squarederror", tree_method="hist",
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=0,
    )
    kf = KFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(cv_model, X_train, y_tr, cv=kf, scoring="r2", n_jobs=-1)
    cv_results.append({
        "Target": tgt,
        "CV_Mean_R²": round(cv_scores.mean(), 4),
        "CV_Std_R²": round(cv_scores.std(), 4),
        "CV_Min_R²": round(cv_scores.min(), 4),
        "CV_Max_R²": round(cv_scores.max(), 4),
    })
    print(f"  {N_CV_FOLDS}-Fold CV R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

print(f"\n{'='*65}")
print("  ALL TARGETS COMPLETE")
print(f"{'='*65}")

## 7  Evaluation Summary

In [ ]:
eval_df = pd.DataFrame(eval_results).set_index("Target")
print("Test Set Metrics:")
display(eval_df)

cv_df = pd.DataFrame(cv_results).set_index("Target")
print("\nCross-Validation Metrics:")
display(cv_df)

print(f"\nMean R²     : {eval_df['R²'].mean():.4f}")
print(f"Mean Adj R² : {eval_df['Adj_R²'].mean():.4f}")
print(f"Mean MAPE   : {eval_df['MAPE (%)'].mean():.2f}%")

In [ ]:
# Predicted vs Actual — 5 panels
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
for ax, tgt, col in zip(axes, TARGET_COLS, COLORS):
    y_pred = models[tgt].predict(X_test)
    y_true = y_test[tgt]
    ax.scatter(y_true, y_pred, alpha=0.50, s=16, color=col, edgecolors="none")
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())
    ax.plot([lo, hi], [lo, hi], "w--", lw=1, alpha=0.5)
    r2 = r2_score(y_true, y_pred)
    ax.set_title(f"{tgt}\nR² = {r2:.4f}", fontsize=9)
    ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
plt.suptitle("Predicted vs Actual — Test Set", fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Residuals distribution
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
for ax, tgt, col in zip(axes, TARGET_COLS, COLORS):
    y_pred = models[tgt].predict(X_test)
    res = y_pred - y_test[tgt]
    ax.hist(res, bins=28, color=col + "aa", edgecolor="none")
    ax.axvline(0, color="white", lw=1, linestyle="--")
    ax.set_title(f"{tgt}\nμ={res.mean():.2f}  σ={res.std():.2f}", fontsize=9)
    ax.set_xlabel("Predicted − Actual")
plt.suptitle("Residual Distributions — Test Set", fontsize=12)
plt.tight_layout(); plt.show()

## 8  Feature Importance (XGBoost native)

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(24, 6))
for ax, tgt, col in zip(axes, TARGET_COLS, COLORS):
    imp = pd.Series(models[tgt].feature_importances_, index=FEATURE_COLS).nlargest(14)
    imp.sort_values().plot.barh(ax=ax, color=col + "cc")
    ax.set_title(tgt, fontsize=10)
    ax.set_xlabel("Importance")
    ax.tick_params(labelsize=7)
plt.suptitle("XGBoost Feature Importance — Top 14 per Target", fontsize=12)
plt.tight_layout(); plt.show()

## 9  SHAP Analysis

In [ ]:
shap_importance = {}
for tgt in TARGET_COLS:
    explainer = shap.TreeExplainer(models[tgt])
    sv = explainer.shap_values(X_test)
    mean_abs = np.abs(sv).mean(axis=0)
    shap_importance[tgt] = {
        feat: round(float(val), 4)
        for feat, val in sorted(zip(FEATURE_COLS, mean_abs), key=lambda x: x[1], reverse=True)[:15]
    }
print("SHAP values computed.")

In [ ]:
# SHAP beeswarm plots — one per target
for tgt in TARGET_COLS:
    explainer = shap.TreeExplainer(models[tgt])
    sv = explainer.shap_values(X_test)
    shap.summary_plot(sv, X_test, feature_names=FEATURE_COLS,
                      max_display=14, show=False, plot_size=(9, 5))
    plt.title(f"SHAP Summary — {tgt}", fontsize=11)
    plt.tight_layout(); plt.show()

In [ ]:
# Global SHAP bar chart (mean |SHAP| averaged across all 5 targets)
all_mean_shap = []
for tgt in TARGET_COLS:
    explainer = shap.TreeExplainer(models[tgt])
    sv = explainer.shap_values(X_test)
    all_mean_shap.append(np.abs(sv).mean(axis=0))
global_shap = pd.Series(np.mean(all_mean_shap, axis=0), index=FEATURE_COLS).nlargest(18).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
global_shap.plot.barh(ax=ax, color="#c9a56bcc")
ax.set_xlabel("Mean |SHAP value| (averaged over all targets)")
ax.set_title("Global Feature Importance via SHAP", fontsize=11)
plt.tight_layout(); plt.show()

## 10  Partial Dependence Plots

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

pdp_features = ["C", "Fe", "Carbon_Equiv", "Cooling_Rate_Est", "HT_Temp_C", "Tempering_Temp_C"]
pdp_features = [f for f in pdp_features if f in X_train.columns]

for tgt in TARGET_COLS:
    fig, axes = plt.subplots(1, len(pdp_features), figsize=(18, 3))
    PartialDependenceDisplay.from_estimator(
        models[tgt], X_train, features=pdp_features, ax=axes,
        line_kw={"color": "#c9a56b", "linewidth": 2.0},
    )
    fig.suptitle(f"Partial Dependence — {tgt}", fontsize=11, y=1.04)
    plt.tight_layout(); plt.show()

## 11  Save Models & Metadata

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Per-target XGBoost JSON files
for tgt, model in models.items():
    fname = os.path.join(OUTPUT_DIR, f"xgb_{tgt.lower()}.json")
    model.save_model(fname)
    print(f"Saved  {fname}")

# Also save to root directory for legacy compatibility
for tgt, model in models.items():
    fname = f"xgb_{tgt.lower()}.json"
    model.save_model(fname)

# Training ranges
training_ranges = {
    col: {"min": float(X_train[col].min()), "max": float(X_train[col].max())}
    for col in FEATURE_COLS
}

# Metadata JSON
metadata = {
    "model_version": "v3",
    "trained_at": datetime.now().isoformat(),
    "features": FEATURE_COLS,
    "targets": TARGET_COLS,
    "training_ranges": training_ranges,
    "model_metrics": eval_results,
    "cv_metrics": cv_results,
    "best_params": best_params,
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "n_features": len(FEATURE_COLS),
    "optuna_trials": OPTUNA_TRIALS,
    "cv_folds": N_CV_FOLDS,
    "early_stopping_rounds": EARLY_STOPPING,
}

for path in [os.path.join(OUTPUT_DIR, "model_metrics.json"), "model_metrics.json"]:
    with open(path, "w") as f:
        json.dump(metadata, f, indent=2)
    print(f"Saved  {path}")

# SHAP importance
with open(os.path.join(OUTPUT_DIR, "shap_importance.json"), "w") as f:
    json.dump(shap_importance, f, indent=2)
print("Saved  models/shap_importance.json")

# Feature order
with open(os.path.join(OUTPUT_DIR, "feature_order.json"), "w") as f:
    json.dump(FEATURE_COLS, f)
print("Saved  models/feature_order.json")

## 12  Example Prediction

**AISI 4140 analogue** — Cr-Mo alloy steel, Quench & Temper

In [ ]:
# Build input for AISI 4140-type steel
ex = {
    "C":0.40, "Si":0.25, "Mn":0.85, "P":0.010, "S":0.010,
    "Ni":0.20, "Cr":1.05, "Cu":0.10, "Mo":0.20,
    "Fe": 100.0 - (0.40+0.25+0.85+0.010+0.010+0.20+1.05+0.10+0.20),
    "HT_Temp_C":850.0, "Soaking_Time_min":45.0,
    "Tempering_Temp_C":550.0, "Tempering_Time_min":120.0,
    "Process_Quench_Temper":1, "Process_Normalizing":0,
    "Process_Annealing":0, "Process_Stress_Relief":0,
    "Cooling_Medium_Oil":1, "Cooling_Medium_Water":0,
    "Cooling_Medium_Polymer":0, "Cooling_Medium_Air":0,
    "Cooling_Medium_Furnace":0, "Cooling_Medium_Salt Bath":0,
}
# Engineered features
c, mn, si, ni, cr, mo, cu = ex["C"], ex["Mn"], ex["Si"], ex["Ni"], ex["Cr"], ex["Mo"], ex["Cu"]
ex["Carbon_Equiv"]    = c + mn/6 + si/24 + ni/40 + cr/5 + mo/4 + cu/15
ex["A3_Temp_C"]       = 912 - 203*np.sqrt(c) - 30*mn + 44.7*si - 15.2*ni + 31.5*mo
ex["Delta_HT_A3"]     = ex["HT_Temp_C"] - ex["A3_Temp_C"]
t_h                   = ex["Tempering_Time_min"] / 60.0
ex["Hollomon_Jaffe"]  = (ex["Tempering_Temp_C"] + 273.15) * (18 + np.log10(t_h))
ex["C_x_Cr"]          = c * cr
ex["Cooling_Rate_Est"]= 25.0  # Oil
ex["Total_Alloy"]     = c + si + mn + ni + cr + mo + cu

X_ex = pd.DataFrame([ex])[FEATURE_COLS]

print("Prediction — AISI 4140 (Q&T: 850°C / Oil / 550°C × 120 min temper)")
print("-" * 55)
for tgt in TARGET_COLS:
    val = models[tgt].predict(X_ex)[0]
    print(f"  {tgt:<20s}:  {val:>8.1f}")

print("\nEffect of tempering temperature on properties:")
for t_temp in [200, 300, 400, 500, 600]:
    ex2 = ex.copy()
    ex2["Tempering_Temp_C"] = float(t_temp)
    t_h2 = ex2["Tempering_Time_min"] / 60.0
    ex2["Hollomon_Jaffe"] = (t_temp + 273.15) * (18 + np.log10(t_h2))
    X2 = pd.DataFrame([ex2])[FEATURE_COLS]
    preds = {tgt: models[tgt].predict(X2)[0] for tgt in TARGET_COLS}
    print(f"  T_temper={t_temp:>4d}°C  →  "
          f"Tensile={preds['Tensile_MPa']:>7.1f}  HB={preds['Hardness_HB']:>6.1f}  "
          f"Elong={preds['Elongation_pct']:>5.2f}%")

## 13  Final Summary

In [ ]:
print("=" * 65)
print("  MAST IQ — Training Pipeline v3 Complete")
print("=" * 65)
print(f"  Dataset      : {len(X)} samples, {len(FEATURE_COLS)} features (incl. Fe)")
print(f"  Split        : {len(X_train)} train / {len(X_test)} test")
print(f"  Optimization : Optuna {OPTUNA_TRIALS} trials × {N_CV_FOLDS}-fold CV per target")
print(f"  Early stop   : {EARLY_STOPPING} rounds")
print(f"")
print(f"  Mean Test R²    : {eval_df['R²'].mean():.4f}")
print(f"  Mean Adj R²     : {eval_df['Adj_R²'].mean():.4f}")
print(f"  Mean MAPE       : {eval_df['MAPE (%)'].mean():.2f}%")
print(f"  Mean CV R²      : {cv_df['CV_Mean_R²'].mean():.4f}")
print(f"")
print(f"  Files saved to: {OUTPUT_DIR}/")
print(f"    - xgb_<target>.json  ×5  (per-target XGBoost models)")
print(f"    - model_metrics.json      (metadata + evaluation)")
print(f"    - shap_importance.json    (SHAP feature importance)")
print(f"    - feature_order.json      (canonical feature order)")
print("=" * 65)

---

### Files to download after training:

Copy these back to your local project's `models/` directory and root:

| File | Location |
|------|----------|
| `xgb_tensile_mpa.json` | `models/` and root |
| `xgb_yield_mpa.json` | `models/` and root |
| `xgb_hardness_hb.json` | `models/` and root |
| `xgb_elongation_pct.json` | `models/` and root |
| `xgb_fatigue_mpa.json` | `models/` and root |
| `model_metrics.json` | `models/` and root |
| `shap_importance.json` | `models/` |
| `feature_order.json` | `models/` |